# *QOT Estimator Basic Usage Example*

> This notebook demonstrates the basic usage of the QOT estimator package for optical link.

In [67]:
# Import Dependencies
import sys
from pathlib import Path
import os
import numpy as np
import pandas as pd
import json
from scipy.io import loadmat

# Ignore warnings to have clean cell outputs
import warnings
warnings.filterwarnings("ignore")

### *Create results directory if it doesn't exist*

In [68]:
# Get the current working directory (where the notebook is running)
base_dir = Path.cwd()

# Define the results directory
results_dir = base_dir.parent / "results" / "single_link"

# Create it if it doesn't exist
results_dir.mkdir(exist_ok=True)

print(f"Results will be saved in: {results_dir}")

Results will be saved in: d:\Projects\CoreLight\Code\CFM_versions\CFM\results\single_link


In [69]:
sys.path.append(os.path.abspath(base_dir.parent / 'src'))
from CFM.core.network import Link, LinkParameters
from CFM.core.band import Band, OpticalParameters
from CFM.core.edfa import EdfaConfig
from CFM.core.qot_estimator import ParameterBuilder, ISRSSolver, NLISolver, ASESolver, OSNRCalculator, OSNRCalculatorV2
from CFM.core.post_process import GSNRPlotter
from CFM.utils.build_alpha_db import build_alpha_for_band

In [70]:
# Get the current working directory (where the notebook is running)
base_dir = Path.cwd()

# Define the results directory
results_dir = base_dir.parent / "results" / "single_link"

# Create it if it doesn't exist
results_dir.mkdir(exist_ok=True)

print(f"Results will be saved in: {results_dir}")

Results will be saved in: d:\Projects\CoreLight\Code\CFM_versions\CFM\results\single_link


# *QOT Estimation*

### *Create spectrum*

In [71]:
my_band_l_params =  OpticalParameters(Rs_mat = 52e9)
my_band_l = Band(
    name='l',
    start_freq = (1.845393470967740e+2 - 0.075/2), # THz
    end_freq = (1.904643470967740e+2 + 0.075/2), # THz
    opt_params = my_band_l_params,
    channel_spacing = 0.075 # THz
    )
spectrum_my_l = my_band_l.calc_spectrum() + 0.075*0.5

my_band_c_params =  OpticalParameters(Rs_mat = 52e9)
my_band_c = Band(
    name='c',
    start_freq = (1.909143470967740e+2 - 0.075/2), # THz
    end_freq = (1.968393470967740e+2 + 0.075/2), # THz
    opt_params = my_band_c_params,
    channel_spacing = 0.075 # THz
    )
spectrum_my_c = my_band_c.calc_spectrum() + 0.075*0.5

my_band_s_part1_params =  OpticalParameters(Rs_mat = 52e9)
my_band_s_part1 = Band(
    name='s',
    start_freq = (1.972893470967740e+2 - 0.075/2), # THz
    end_freq = (2.035143470967740e+2 + 0.075/2), # THz
    opt_params = my_band_s_part1_params,
    channel_spacing = 0.075 # THz
    )
spectrum_my_s_part1 = my_band_s_part1.calc_spectrum() + 0.075*0.5


my_band_s_part2_params =  OpticalParameters(Rs_mat = 52e9)
my_band_s_part2 = Band(
    name='s',
    start_freq = (2.035893870967740e+2 - 0.075/2), # THz
    end_freq = (2.053143870967740e+2 + 0.075/2), # THz
    opt_params = my_band_s_part2_params,
    channel_spacing = 0.075 # THz
    )
spectrum_my_s_part2 = my_band_s_part2.calc_spectrum() + 0.075*0.5

grid_center = np.concatenate((spectrum_my_s_part2 , spectrum_my_s_part1, spectrum_my_c , spectrum_my_l))
grid_center = grid_center[::-1]*1e12
bands = [my_band_l, my_band_c, my_band_s_part1, my_band_s_part2]

In [72]:
alpha_dB_LCS_268channels = loadmat('.././data/alpha_dB_LCS_268channels.mat')
alpha_dB_LCS = alpha_dB_LCS_268channels['alpha_dB_LCS']

## *Multi Span*

### *Create fiber link*

In [73]:
my_link_params = LinkParameters()
my_link_l1 = Link(
  name='l1',
  length=[70, 70],
  num_span=2,
  num_amp=2,
  link_params=my_link_params)

In [74]:
booster_amp = EdfaConfig(gain_target=15)
span_edfas = [
    EdfaConfig(gain_target=12.0, tilt_target=0.0),  # Span 1
    EdfaConfig(gain_target=12.0, tilt_target=0.0),                   # Span 2
    # EdfaConfig(gain_target=12.0, tilt_target=0.0, out_voa=0.0)       # Span 3
]

#### *FLP Example*

In [122]:
my_link_SNR_Param = ParameterBuilder(
  link=my_link_l1,
  bands=bands,
  P_in = [-3., 2.],
  grid_center=grid_center,
  P_in_is_tx_power = True,
  alpha_dB_LCS=alpha_dB_LCS
)

In [123]:
my_link_ISRS = ISRSSolver(my_link_SNR_Param, 'FLP')

In [124]:
p, s1, s2, sig = my_link_ISRS.solve()

In [125]:
p

array([[-3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3.,
        -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3.,
        -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3.,
        -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3.,
        -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3.,
        -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3.,
        -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3.,
        -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3.,
        -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3.,
        -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3.,
        -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3.,
        -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3.,
        -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3.,
        -3., -3., -3., -3., -3., -3., -3., -3., -3.

In [126]:
my_link_ISRS = ISRSSolver(my_link_SNR_Param, 'FLP')

In [127]:
my_link_NLI = NLISolver(my_link_SNR_Param, s1, s2, sig)

In [128]:
nli = my_link_NLI.solve()

In [129]:
my_link_ASE = ASESolver(my_link_SNR_Param) 

In [130]:
ase = my_link_ASE.solve()

In [131]:
my_link_OSNR = OSNRCalculator(my_link_SNR_Param, ase, nli)

In [132]:
on, oa, ot = my_link_OSNR.compute()

In [134]:
ot[1]

array([28.49263934, 28.27072937, 28.16859898, 28.10203337, 28.05262035,
       28.01326642, 27.98050408, 27.9523861 , 27.92749437, 27.90347293,
       27.883563  , 27.86584251, 27.84961426, 27.83484177, 27.82126215,
       27.80869669, 27.79701367, 27.78614478, 27.77591248, 27.76629684,
       27.75721587, 27.74856609, 27.74020678, 27.73197748, 27.72181318,
       27.71425102, 27.70734145, 27.70080517, 27.69448463, 27.68849044,
       27.68263292, 27.67692501, 27.67136514, 27.66592526, 27.66045948,
       27.65496923, 27.6484809 , 27.64309191, 27.63786978, 27.6327475 ,
       27.62759754, 27.62253227, 27.61740874, 27.61225868, 27.60704137,
       27.60183827, 27.59649662, 27.59093144, 27.58408218, 27.57842292,
       27.57294947, 27.56766797, 27.56226993, 27.55685107, 27.55147171,
       27.54606353, 27.54059912, 27.53520087, 27.52965467, 27.52361443,
       27.51823571, 27.5129951 , 27.50778437, 27.50274405, 27.49774545,
       27.49263229, 27.48802119, 27.48376816, 27.47974252, 27.47

In [136]:
curves = [
    {"name": "GSNR_total", "values": ot, "color": "blue"},
    {"name": "GSNR_linear", "values": oa, "color": "green"},
    {"name": "GSNR_nonlinear", "values": on, "color": "red"},
]

plotter = GSNRPlotter(my_link_SNR_Param, curves, bands=bands)

fig = plotter.plot(N_s_max=0, matlab_indexing=True)
fig.show()


In [41]:
my_link_OSNRV2 = OSNRCalculatorV2(my_link_SNR_Param, ase, nli)

In [42]:
res = my_link_OSNRV2.compute()

In [43]:
res['OSNR_NLI_ASE_dB']

array([[28.97446179, 28.9364709 , 28.92115489, 28.9125613 , 28.9072919 ,
        28.90394845, 28.90181005, 28.90053612, 28.89989289, 28.89977541,
        28.8999811 , 28.90038198, 28.90223197, 28.9043878 , 28.90696481,
        28.90988734, 28.91310862, 28.91651017, 28.92035852, 28.924236  ,
        28.92822903, 28.93249258, 28.93692956, 28.94136457, 28.94598795,
        28.95071765, 28.95543388, 28.96024775, 28.9650875 , 28.9695883 ,
        28.97439825, 28.9792328 , 28.98405522, 28.98873448, 28.9933871 ,
        28.99785324, 29.00241347, 29.00686834, 29.01124846, 29.01546728,
        29.01970547, 29.02377324, 29.02780761, 29.03172547, 29.03558301,
        29.03917561, 29.0427715 , 29.04627364, 29.04970036, 29.05304999,
        29.0562845 , 29.05918679, 29.06222623, 29.06517132, 29.06798491,
        29.07078762, 29.07350556, 29.07610486, 29.0787489 , 29.08134001,
        29.08384557, 29.08642421, 29.08917584, 29.09188654, 29.09480844,
        29.09796101, 29.10140901, 29.10520087, 29.1

#### *FLP Example*

In [44]:
my_link_SNR_Param = ParameterBuilder(
  link=my_link_l1,
  bands=bands,
  P_in = 2.,
  grid_center=grid_center,
  alpha_dB_LCS=alpha_dB_LCS
)

In [45]:
my_link_ISRS = ISRSSolver(my_link_SNR_Param, 'FLP')

In [46]:
s1, s2, sig = my_link_ISRS.solve()

ValueError: too many values to unpack (expected 3)

In [ ]:
my_link_NLI = NLISolver(my_link_SNR_Param, s1, s2, sig)

In [ ]:
nli = my_link_NLI.solve()

In [ ]:
my_link_ASE = ASESolver(my_link_SNR_Param) 

In [ ]:
ase = my_link_ASE.solve()

In [ ]:
my_link_OSNR = OSNRCalculator(my_link_SNR_Param, ase, nli)

In [ ]:
on, oa, ot = my_link_OSNR.compute()

In [ ]:
curves = [
    {"name": "GSNR_total", "values": ot, "color": "blue"},
    {"name": "GSNR_linear", "values": oa, "color": "green"},
    {"name": "GSNR_nonlinear", "values": on, "color": "red"},
]

plotter = GSNRPlotter(my_link_SNR_Param, curves, bands=bands)

fig = plotter.plot(N_s_max=1, matlab_indexing=True)
fig.show()


In [ ]:
ot

array([[32.78196928, 32.37875334, 32.19855991, 32.0827865 , 31.99766468,
        31.93042838, 31.87485255, 31.82740224, 31.7856893 , 31.74566866,
        31.71271161, 31.68357484, 31.65755617, 31.63418412, 31.61301624,
        31.59379313, 31.57627913, 31.56028529, 31.54565672, 31.53222751,
        31.51983345, 31.50833735, 31.49756155, 31.48700663, 31.47349483,
        31.46445004, 31.45661797, 31.44954371, 31.44309694, 31.43725206,
        31.43191119, 31.42708021, 31.4226912 , 31.4186705 , 31.41490513,
        31.41118211, 31.40597874, 31.4027126 , 31.39984338, 31.3971641 ,
        31.39452788, 31.39195793, 31.38935701, 31.38673872, 31.38404929,
        31.38133606, 31.37845024, 31.37520109, 31.36988984, 31.36660458,
        31.36366173, 31.36092809, 31.35813401, 31.35537932, 31.35270508,
        31.35006435, 31.34743827, 31.34491756, 31.342257  , 31.33883129,
        31.33658067, 31.33471199, 31.33306739, 31.33180867, 31.33079372,
        31.32973297, 31.32969555, 31.33040484, 31.3